### 11376/8890 Computer Vision and Image Analysis - Computer Laboratory
# Week 11 - Object Recognition (with Bounding Boxes)
[25/Apr/2026 Dr Min Wang]

## Learning Objectives
By the end of this lab, building on your knowledge from Week 10 on classical and deep learning methods, you should be able to:
- Apply previously learned classification methods (handcrafted features+SVM and CNNs) to a real-world fine-grained dataset CUB-200-2011.
- Extend model pipelines to more complex data settings, including handling annotations and performing bounding box–based region extraction.
- Implement and compare different feature representations, including handcrafted features and learned features from CNNs.
- Evaluate and interpret model performance, using metrics such as accuracy and confusion matrices, with a focus on reasoning rather than just numerical results, and using cross validation.
- Analyse the impact of input design choices, particularly comparing whole-image classification versus object-region (bounding box) classification.

## Overview
In this lab, we will investigate image classification using a subset of the CUB-200-2011 bird dataset. Our sample dataset contains images of 20 bird species, with annotations including class labels and bounding boxes.
We will compare four experimental settings:
1.	Whole image classification using handcrafted features and machine learning 
2.	Whole image classification using deep learning 
3.	Bounding-box region classification using handcrafted features and machine learning 
4.	Bounding-box region classification using deep learning 
The goal is to understand how feature representation, model choice, and object localisation affect classification performance.

Please follow the Week 11 Lab Note, refer to Week 10 Lab materials and try to work this out independently first. You may then check this provided example code to compare your approach and confirm whether you are on the right track 




## Dataset Loading + Train/Test Set Creation 
Each file provides partial information about the dataset:
- images.txt → where the image is stored
- image_class_labels.txt → what class it belongs to
- bounding_boxes.txt → where the bird is located
- train_test_split.txt → whether it is used for training or testing (1-training, 0-testing)

We combine them into one unified table.

After merging, each row contains: 
image_id | image_path | class_id | x | y | width | height | is_train

This gives us everything needed for all four experiments:
- classification label
- bounding box for cropping
- train/test assignment


In [1]:
import os
import pandas as pd

# Root directory of your dataset
DATA_DIR = "CUB_200_2011_Subset20classes"
# Path to the image folder
IMAGE_DIR = os.path.join(DATA_DIR, "images")

####################################
## Step 1: Load Metadata Files
####################################
# images.txt: maps image_id → relative file path
images_df = pd.read_csv(
    os.path.join(DATA_DIR, "images.txt"),
    sep=" ",
    names=["image_id", "image_path"]
)

# image_class_labels.txt: maps image_id → class label
labels_df = pd.read_csv(
    os.path.join(DATA_DIR, "image_class_labels.txt"),
    sep=" ",
    names=["image_id", "class_id"]
)

# bounding_boxes.txt: provides object localisation (bird region)
bbox_df = pd.read_csv(
    os.path.join(DATA_DIR, "bounding_boxes.txt"),
    sep=" ",
    names=["image_id", "x", "y", "width", "height"]
)

# train_test_split.txt: indicates whether image is training or testing
split_df = pd.read_csv(
    os.path.join(DATA_DIR, "train_test_split.txt"),
    sep=" ",
    names=["image_id", "is_train"]
)

####################################
## Step 2: Merge All Information
####################################
data = images_df.merge(labels_df, on="image_id")
data = data.merge(bbox_df, on="image_id")
data = data.merge(split_df, on="image_id")

####################################
## Step 3: Remap class labels to 0–19
####################################
# Get unique class IDs (e.g., [3, 7, 15, ...])
unique_classes = sorted(data["class_id"].unique())

# Create mapping → convert labels to 0–(N-1)
class_mapping = {old: new for new, old in enumerate(unique_classes)}

# Apply mapping
data["label"] = data["class_id"].map(class_mapping)

####################################
## Step 4: Build Full Image Paths
####################################
data["full_path"] = data["image_path"].apply(
    lambda x: os.path.join(IMAGE_DIR, x)
)

####################################
## Step 5: Split into Training and Testing Sets
####################################
train_df = data[data["is_train"] == 1].reset_index(drop=True)
test_df = data[data["is_train"] == 0].reset_index(drop=True)

train_paths = train_df["full_path"].values
train_labels = train_df["label"].values
train_bboxes = train_df[["x", "y", "width", "height"]].values

test_paths = test_df["full_path"].values
test_labels = test_df["label"].values
test_bboxes = test_df[["x", "y", "width", "height"]].values

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))
print("Number of classes:", data["label"].nunique())

Training samples: 892
Testing samples: 223
Number of classes: 20


## Experiment 1: Whole Image + HOG + SVM (Example Code)


In [2]:
from skimage.feature import hog
from skimage.color import rgb2gray
from skimage.transform import resize
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from PIL import Image
import numpy as np

# This function extracts HOG feature from an image
def extract_hog(image_path):
    # Load image and convert to RGB
    image = Image.open(image_path).convert("RGB")
    
    # Resize to fixed size for consistent feature extraction
    image = resize(np.array(image), (128, 128))
    
    # Convert to grayscale (HOG works on intensity gradients)
    gray = rgb2gray(image)

    # Extract HOG features (edge/shape descriptors)
    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2)
    )
    return features

# This function builds the dataset (X, y)
def build_dataset(paths, labels):
    X, y = [], []
    for p, l in zip(paths, labels):
        # Extract feature for each image
        X.append(extract_hog(p))
        y.append(l)
    return np.array(X), np.array(y)


# Build training and testing feature sets
X_train, y_train = build_dataset(train_paths, train_labels)
X_test, y_test = build_dataset(test_paths, test_labels)

# Train SVM classifier (linear kernel works well for HOG)
model = SVC(kernel="linear")
model.fit(X_train, y_train)

# Predict on test data
y_pred = model.predict(X_test)

# Evaluate performance
print("Experiment 1 (Whole Image + HOG + SVM) Accuracy:", accuracy_score(y_test, y_pred))

# You should also calculate the confusion matrix and visualise some correct/incorrect samples to reason the results.

Experiment 1 (Whole Image + HOG + SVM) Accuracy: 0.16143497757847533


## Experiment 2: Whole Image + Deep Learning
### (a) CNN from Scratch (Example Code)

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 20

def load_image(path, label):
    # Read image file
    img = tf.io.read_file(path)
    
    # Decode JPEG → tensor
    img = tf.image.decode_jpeg(img, channels=3)
    
    # Resize to CNN input size
    img = tf.image.resize(img, IMG_SIZE)
    
    # Normalize pixel values to [0,1]
    img = img / 255.0
    
    return img, label


# Build TensorFlow dataset
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.map(load_image).shuffle(500).batch(BATCH_SIZE)

test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = test_ds.map(load_image).batch(BATCH_SIZE)


def cnn_scratch():
    # Define simple CNN architecture
    model = models.Sequential([
        layers.Input((224,224,3)),

        # Learn low-level features (edges, textures)
        layers.Conv2D(32, 3, activation='relu'),
        layers.MaxPooling2D(),

        # Learn more complex patterns
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),

        # Learn higher-level object structures
        layers.Conv2D(128, 3, activation='relu'),
        layers.MaxPooling2D(),

        # Flatten feature maps to vector
        layers.Flatten(),

        # Fully connected layer for classification
        layers.Dense(128, activation='relu'),

        # Output layer (20 bird classes)
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    # Compile model
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


model = cnn_scratch()

# Train CNN
model.fit(train_ds, epochs=10)

# Evaluate performance
print("Experiment 2A (Whole Image + CNN from Scratch) Accuracy:", model.evaluate(test_ds)[1])

Epoch 1/10


InvalidArgumentError: Graph execution error:

Detected at node ReadFile defined at (most recent call last):
<stack traces unavailable>
Error in user-defined function passed to MapDataset:1 transformation with iterator: Iterator::Root::Prefetch::BatchV2::Shuffle::ParallelMapV2: NewRandomAccessFile failed to Create/Open: CUB_200_2011_Subset20classes\images\001.Black_footed_Albatross/Black_Footed_Albatross_0046_18.jpg : The filename, directory name, or volume label syntax is incorrect.
; no protocol option
	 [[{{node ReadFile}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_1981]

## Experiment 2: Whole Image + Deep Learning (Example Code)
### (B) Transfer Learning
In this example, we will use the pre-trained model MobileNetV2 and apply transfer learning to our problem.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def load_image_transfer(path, label):
    # Load and decode image
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)

    # Resize to model input size
    img = tf.image.resize(img, IMG_SIZE)

    # Apply pretrained model normalization
    img = preprocess_input(img)

    return img, label


# Create dataset
train_ds_t = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds_t = train_ds_t.map(load_image_transfer).batch(BATCH_SIZE)

test_ds_t = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds_t = test_ds_t.map(load_image_transfer).batch(BATCH_SIZE)


def transfer_model():
    # Load pretrained model (feature extractor)
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

    # Freeze pretrained layers (do not update weights)
    base.trainable = False

    # Add custom classification head
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(base.input, out)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


model = transfer_model()

# Train only classifier head
model.fit(train_ds_t, epochs=10)

print("Experiment 2B (Whole Image + Transfer Learning) Accuracy:", model.evaluate(test_ds_t)[1])

Epoch 1/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 127ms/step - accuracy: 0.0179 - loss: 6.7389
Epoch 2/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - accuracy: 0.1211 - loss: 3.2057
Epoch 3/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.2253 - loss: 2.5842
Epoch 4/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.2859 - loss: 2.2641
Epoch 5/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.3969 - loss: 1.8741
Epoch 6/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - accuracy: 0.4697 - loss: 1.6418
Epoch 7/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - accuracy: 0.6155 - loss: 1.2645
Epoch 8/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - accuracy: 0.6424 - loss: 1.1264
Epoch 9/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.7567 - loss: 0.8841
Epoch 10/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - accuracy: 0.7937 - loss: 0.7155
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 167ms/step - accuracy: 0.7354 - loss: 0.8308
Experiment 2B (Whole Image + Transfer Learning) Accuracy: 0.735426

## Experiment 3: Bounding Box + HOG + SVM (Example Code)


In [ ]:
def extract_hog_bbox(path, bbox):
    x, y, w, h = bbox

    # Load image
    img = Image.open(path).convert("RGB")

    # Crop only the object region (bird)
    crop = img.crop((x, y, x+w, y+h))

    # Resize for consistency
    crop = resize(np.array(crop), (128,128))

    # Convert to grayscale
    gray = rgb2gray(crop)

    # Extract HOG features from cropped region
    return hog(gray, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2))


def build_bbox_dataset(paths, labels, bboxes):
    X, y = [], []
    for p, l, b in zip(paths, labels, bboxes):
        X.append(extract_hog_bbox(p, b))
        y.append(l)
    return np.array(X), np.array(y)


# Build dataset using cropped images
X_train, y_train = build_bbox_dataset(train_paths, train_labels, train_bboxes)
X_test, y_test = build_bbox_dataset(test_paths, test_labels, test_bboxes)

# Train SVM
model = SVC(kernel="linear")
model.fit(X_train, y_train)

# Evaluate
print("Experiment 3 (Bounding Box + HOG + SVM) Accuracy:", accuracy_score(y_test, model.predict(X_test)))

Experiment 3 (Bounding Box + HOG + SVM) Accuracy: 0.2062780269058296


## Experiment 4: Bounding Box + Deep Learning (Example Code)
### (A) CNN from Scratch


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 20

def load_bbox(path, label, bbox):
    # Load image
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)

    # Get image height and width
    img_shape = tf.shape(img)
    img_h = img_shape[0]
    img_w = img_shape[1]

    # Read bbox values
    x = tf.cast(bbox[0], tf.int32)
    y = tf.cast(bbox[1], tf.int32)
    w = tf.cast(bbox[2], tf.int32)
    h = tf.cast(bbox[3], tf.int32)

    # Make sure x and y are not negative
    x = tf.maximum(x, 0)
    y = tf.maximum(y, 0)

    # Make sure width and height do not exceed image boundary
    w = tf.minimum(w, img_w - x)
    h = tf.minimum(h, img_h - y)

    # Make sure width and height are at least 1
    w = tf.maximum(w, 1)
    h = tf.maximum(h, 1)

    # Crop bird region
    img = tf.image.crop_to_bounding_box(
        img,
        offset_height=y,
        offset_width=x,
        target_height=h,
        target_width=w
    )

    # Resize and normalise
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0

    return img, label


# Build dataset with cropped images
train_ds_bbox = tf.data.Dataset.from_tensor_slices((train_paths, train_labels, train_bboxes))
train_ds_bbox = (
    train_ds_bbox
    .map(load_bbox)
    .shuffle(500)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# Build dataset with cropped images
test_ds_bbox = tf.data.Dataset.from_tensor_slices((test_paths, test_labels, test_bboxes))
test_ds_bbox = (
    test_ds_bbox
    .map(load_bbox)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

model = cnn_scratch()

# Train CNN on cropped images
model.fit(train_ds_bbox, epochs=10)

print("Experiment 4A (Bounding Box + CNN from Scratch) Accuracy:", model.evaluate(test_ds_bbox)[1])


Epoch 1/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.0897 - loss: 3.4634
Epoch 2/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - accuracy: 0.3049 - loss: 2.3796
Epoch 3/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 97ms/step - accuracy: 0.4428 - loss: 1.9075
Epoch 4/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - accuracy: 0.6379 - loss: 1.2423
Epoch 5/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 97ms/step - accuracy: 0.7679 - loss: 0.8335
Epoch 6/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - accuracy: 0.8419 - loss: 0.5160
Epoch 7/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - accuracy: 0.9226 - loss: 0.2964
Epoch 8/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - accuracy: 0.8711 - loss: 0.7059
Epoch 9/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.8655 - loss: 0.5222
Epoch 10/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - accuracy: 0.9372 - loss: 0.2773
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.4215 - loss: 5.0562
Experiment 4A (Bounding Box + CNN from Scratch) Accuracy: 0.4215246737003

## Experiment 4: Bounding Box + Deep Learning (Example Code)
### (B) Transfer Learning

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def load_bbox_transfer(path, label, bbox):
    # First crop + normalize like before
    img, label = load_bbox(path, label, bbox)

    # Convert to pretrained model input format
    img = preprocess_input(img * 255.0)

    return img, label

def transfer_model():
    # Load pretrained model (feature extractor)
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

    # Freeze pretrained layers (do not update weights)
    base.trainable = False

    # Add custom classification head
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(base.input, out)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


train_ds_bbox_t = tf.data.Dataset.from_tensor_slices((train_paths, train_labels, train_bboxes))
train_ds_bbox_t = train_ds_bbox_t.map(lambda p,l,b: load_bbox_transfer(p,l,b)).batch(BATCH_SIZE)

test_ds_bbox_t = tf.data.Dataset.from_tensor_slices((test_paths, test_labels, test_bboxes))
test_ds_bbox_t = test_ds_bbox_t.map(lambda p,l,b: load_bbox_transfer(p,l,b)).batch(BATCH_SIZE)


model = transfer_model()

# Train transfer learning model on cropped images
model.fit(train_ds_bbox_t, epochs=10)

print("Experiment 4B (Bounding Box + Transfer Learning) Accuracy:", model.evaluate(test_ds_bbox_t)[1])

Epoch 1/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 128ms/step - accuracy: 0.0179 - loss: 6.5483
Epoch 2/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - accuracy: 0.1031 - loss: 3.5122
Epoch 3/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - accuracy: 0.2119 - loss: 2.6543
Epoch 4/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - accuracy: 0.3700 - loss: 2.0719
Epoch 5/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - accuracy: 0.5594 - loss: 1.4399
Epoch 6/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - accuracy: 0.6491 - loss: 1.2008
Epoch 7/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - accuracy: 0.7265 - loss: 0.9479
Epoch 8/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 107ms/step - accuracy: 0.7948 - loss: 0.7617
Epoch 9/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - accuracy: 0.8397 - loss: 0.6221
Epoch 10/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - accuracy: 0.8677 - loss: 0.5399
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 158ms/step - accuracy: 0.8117 - loss: 0.7372
Experiment 4B (Bounding Box + Transfer Learning) Accuracy: 0.81165

## Adding Data Augmentation and Cross Validation
Finally, let's look at data augmentation and k-fold cross validation. I will use the Experiment 4B as an example.

Data augmentation artificially increases the diversity of the training data by applying small transformations to images. Instead of collecting more data, we create new variations of existing images. In the example code, we:
- randomly flip the image left to right
- rotates image by a small angle (+-10%)
- randomly zooms in/our slightly

This helps the model generalise better to unseen data, reduces overfitting (model memorising training images), makes the model more robust to real-world variations such as:
- different viewpoints
- slight rotations
- scale changes

It is important to note: data augmentation is applied only during training, not during validation or testing.
- Validation and test results should reflect true model performance
- No artificial transformations should affect evaluation

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 20
EPOCHS = 10
K = 5 # K fold cross validation

################################
## Data augmentation
################################
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),  # flip
    layers.RandomRotation(0.1),  # rotate
    layers.RandomZoom(0.1),  # zoom in/out
])


################################
## Safe Bounding Box Loader for Transfer Learning (this part remains the same)
################################
def load_bbox_transfer(path, label, bbox):
    # Read image file
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)

    # Get image size
    img_shape = tf.shape(img)
    img_h = img_shape[0]
    img_w = img_shape[1]

    # Read bounding box values
    x = tf.cast(bbox[0], tf.int32)
    y = tf.cast(bbox[1], tf.int32)
    w = tf.cast(bbox[2], tf.int32)
    h = tf.cast(bbox[3], tf.int32)

    # Clip bounding box to valid image boundary
    x = tf.maximum(x, 0)
    y = tf.maximum(y, 0)

    w = tf.minimum(w, img_w - x)
    h = tf.minimum(h, img_h - y)

    w = tf.maximum(w, 1)
    h = tf.maximum(h, 1)

    # Crop bird region
    img = tf.image.crop_to_bounding_box(
        img,
        offset_height=y,
        offset_width=x,
        target_height=h,
        target_width=w
    )

    # Resize to MobileNetV2 input size
    img = tf.image.resize(img, IMG_SIZE)

    # Apply MobileNetV2 preprocessing
    img = preprocess_input(img)

    return img, label


################################
## Transfer Learning Model with Augmentation
################################
def create_transfer_model_with_augmentation():
    # Load pretrained MobileNetV2 without the original classifier
    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Freeze pretrained layers
    base_model.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))

    # Apply data augmentation only during training
    x = data_augmentation(inputs)

    # Extract pretrained visual features
    x = base_model(x, training=False)

    # Convert feature maps into a feature vector
    x = layers.GlobalAveragePooling2D()(x)

    # Classification head
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model



################################
## K-Fold Cross-Validation
################################

# Convert to NumPy arrays
all_paths = np.array(train_paths)
all_labels = np.array(train_labels)
all_bboxes = np.array(train_bboxes)

# StratifiedKFold is a variation of K-fold cross-validation that ensures each fold
# has a similar class distribution as the original dataset.
skf = StratifiedKFold(
    n_splits=K,
    shuffle=True,
    random_state=42
)

fold_accuracies = []

# loop over the k folds
for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels)):
    print(f"\n===== Fold {fold + 1}/{K} =====")

    # Split data for current fold
    fold_train_paths = all_paths[train_idx]
    fold_train_labels = all_labels[train_idx]
    fold_train_bboxes = all_bboxes[train_idx]

    fold_val_paths = all_paths[val_idx]
    fold_val_labels = all_labels[val_idx]
    fold_val_bboxes = all_bboxes[val_idx]

    # Training dataset
    train_ds = tf.data.Dataset.from_tensor_slices(
        (fold_train_paths, fold_train_labels, fold_train_bboxes)
    )

    train_ds = (
        train_ds
        .map(load_bbox_transfer)
        .shuffle(500)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    # Validation dataset
    val_ds = tf.data.Dataset.from_tensor_slices(
        (fold_val_paths, fold_val_labels, fold_val_bboxes)
    )

    val_ds = (
        val_ds
        .map(load_bbox_transfer)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    # Important: create a new model for each fold
    model = create_transfer_model_with_augmentation()

    # Train model on this fold
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS
    )

    # Evaluate validation performance
    val_loss, val_acc = model.evaluate(val_ds)

    print(f"Fold {fold + 1} Validation Accuracy: {val_acc:.4f}")

    fold_accuracies.append(val_acc)

print("\nExperiment 4B Cross-validation results")
print("Fold accuracies:", fold_accuracies)
print("Mean validation accuracy:", np.mean(fold_accuracies))
print("Standard deviation:", np.std(fold_accuracies))


===== Fold 1/5 =====
Epoch 1/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 11s 271ms/step - accuracy: 0.1445 - loss: 3.8590 - val_accuracy: 0.4972 - val_loss: 1.6136
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 154ms/step - accuracy: 0.3058 - loss: 3.0206 - val_accuracy: 0.6704 - val_loss: 1.0338
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 149ms/step - accuracy: 0.4306 - loss: 2.3572 - val_accuracy: 0.7318 - val_loss: 0.8142
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 153ms/step - accuracy: 0.5330 - loss: 1.8216 - val_accuracy: 0.8268 - val_loss: 0.5658
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 154ms/step - accuracy: 0.6059 - loss: 1.5566 - val_accuracy: 0.7821 - val_loss: 0.6515
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 152ms/step - accuracy: 0.6410 - loss: 1.3883 - val_accuracy: 0.8492 - val_loss: 0.4683
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 156ms/step - accuracy: 0.6788 - loss: 1.0341 - val_accuracy: 0.8603 - val_loss: 0.4086
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 158ms/step - accuracy: 0.7209 - loss: 0.

## Final Notes: 
In this above example, we did not explicitly distinguish between a validation set and a testing set. However, it is important to clarify that, in best practice, these two sets serve different purposes. A validation set is used during model development to monitor performance and guide decisions such as hyperparameter tuning, while a testing set should remain completely unseen until the very end to provide an unbiased evaluation of the final model. Using the same data for both roles can lead to overly optimistic results, as the model may be indirectly tuned to perform well on that data. Therefore, in more rigorous experimental settings, the dataset should be divided into three parts: training, validation, and testing, to ensure fair and reliable performance assessment.
- Training set → train the model
- Validation set → monitor performance during training and guide hyperparameter tuning
- Testing set → final testing after training

You can think about how to change the above code to have a more rigurous performance assessment.